In [ ]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, Normalizer, StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import silhouette_score

import seaborn as sns
import matplotlib.pyplot as plt

import os, sys
sys.path.append(os.path.abspath("../"))
from src.helper import show_clustered_image

## 1. Load Dataset
> Karena fitur yang mempengaruhi harga harga cetak adalah hitam-putih dan berwarna, maka fitur yang akan digunakan adalah fitur `cmy` dan `k`
> - `cmy` : Penjumlahan persentase penggunaan warna Cyan + Magenta + Yellow (Berwarna)
> - `k`   : Persentase penggunaan warna Black (Hitam-Putih)

In [ ]:
df = pd.read_csv("../outputs/csv/cmyk_of_a_pdf_file_by_dpi.csv")
df = df[(df['dpi'] == 300) & (df['library'] == 'pymupdf')].copy()
df.reset_index(drop=True, inplace=True)
df.sample(5)

In [ ]:
from sklearn.base import ClusterMixin

# Scorer function
def silhouette_scorer(estimator, X):
    """
    Scorer function for silhouette score to be used in RandomizedSearchCV.
    Handles pipeline estimators by accessing the clustering step.
    
    Parameters:
        estimator: clustering model or pipeline (must have a fit_predict method).
        X: Input data.
    
    Returns:
        silhouette score.
    """
    # Jika estimator adalah pipeline, ambil model clustering dari pipeline
    if hasattr(estimator, "named_steps"):
        clustering_model = estimator.named_steps['algo']  # Ambil langkah 'algo' dari Pipeline
    else:
        clustering_model = estimator

    # Pastikan clustering_model adalah model clustering
    if isinstance(clustering_model, ClusterMixin):
        labels = clustering_model.fit_predict(X)
        return silhouette_score(X, labels)
    else:
        raise ValueError("The estimator must be a clustering model (like KMeans).")

In [ ]:

col_transformer = ColumnTransformer([
    ("scaling", RobustScaler(), ['cmy'])
])

model = Pipeline([
    ('prep', col_transformer),
    ('algo', AgglomerativeClustering(n_clusters=40))
])

model.fit(df)

> Jumlah cluster mula - mula adalah 10, nanti akan digabung menjadi beberapa cluster pada proses anotasi

In [ ]:
y_pred = model.fit_predict(df)
df['label'] = y_pred
df.sample(5)

In [ ]:
plt.figure(figsize=(18,8))
sns.scatterplot(df, x='cmy', y='k', hue='label', palette='bright')
plt.xlabel("Cyan, Magenta, Yellow (%)")
plt.ylabel("Black (%)")
plt.title("Perbandingan Persentase C+M+Y dan K")
plt.show()

**Mengapa Persentasenya ada yang > 100%?**
> Persentase maksimal untuk masing - masing warna adalah 100%. Karena terdapat 3 warna, maka **penjumlahan ketiganya maksimal adalah 300%** yang berarti menghasilkan warna yang sangat gelap sebagai hasil dari pencampuran 3 warna tersebut.
> Semakin banyak persentase penjumlahan CMYK, maka warna yang dihasilkan akan semakin gelap

### 1.1 Menambahkan kolom `sum` untuk memudahkan proses anotasi
> Meskipun warnanya dipisah menjadi Hitam-Putih dan Cyan-Magenta-Yellow, persentase keseluruhan penggunaan tinta adalah penjumlahan antara keduanya. Semakin banyak persentase warna yang digunakan, maka akan semakin mahal biaya cetaknya. Oleh karena itu perlu ditambahkan kolom baru yang merupakan penjumlahan keduanya yaitu kolom `sum`

In [ ]:
df[['cmyk', 'label']].groupby("label").agg(['min', 'max', 'mean']).sort_values(('cmyk', 'max'))

### 1.2 Menampilkan Halaman Setiap Cluster

In [ ]:
for i in df.sort_values("cmyk").index:
    print(f"{'-'*25}Label {i}{'-'*25}")
    show_clustered_image(df, i)

## 2. Label Annotation
> Melabeli harga dari 15 cluster menjadi 5 cluster harga dalam Rupiah (IDR), yaitu
> - 500
> - 750
> - 1000
> - 1500
> - 2000

In [ ]:
label_annotation = {
     27: 500,
     34: 500,
     25: 500,
     13: 500,
     29: 500,
     37: 500,
     36: 500,
     21: 750,
     14: 750,
     18: 750,
     33: 1000,
     38: 1000,
     19: 1000,
     2: 1000,
     39: 1000,
     11: 1000,
     30: 1000,
     9: 1000,
     0: 1500,
     8: 1500,
     1: 1500,
     3: 1500,
     15: 1500,
     12: 1500,
     5: 1500,
     16: 1500,
     17: 1500,
     22: 1500,
     26: 2000,
     23: 2000,
     7: 2000,
     24: 2000,
     31: 2000,
     10: 2000,
     6: 2000,
     4: 2000,
     20: 2000,
     32: 2000,
     35: 2000,
     28: 2000,
}
# len(label_annotation)
df['label'] = df['label'].replace(label_annotation)
df

### 2.1 Menyimpan hasil 40 cluster
> Pada saat inferensi, sangat mungkin terjadi `data drift` maupun `concept drift` yang dapat mengurangi accuracy dan relevansi model. Kita dapat melakukan adjustment harga dengan acuan `old_cluster` yang merupakan hasil dari 40 cluster sebelum digabungkan menjadi 5 cluster.

In [ ]:
df['old_label'] = res
df

## 3. Hasil Annotasi
> - Anotasi cluster dengan harga cetak
> - 15 Cluster menjadi 5 Cluster

In [ ]:
plt.figure(figsize=(18,8))
sns.scatterplot(df, x='cmy', y='k', hue='label', palette='bright')
plt.xlabel("Cyan, Magenta, Yellow (%)")
plt.ylabel("Black (%)")
plt.title("Perbandingan Persentase C+M+Y dan K")
plt.show()

## 4. Menampilkan Halaman Berdasarkan Cluster Harga

In [ ]:
for i in sorted(df['label'].unique()):
    print("Harga: ", i)
    show_clustered_image(df, i)

## 5. Menyimpan Data Hasil Clustering

In [ ]:
df.to_csv("../datasets/clustered_cmy_k_agglomerative_300dpi.csv", index=False)

## 6. Menyimpan Model Clustering

In [ ]:
import pickle

pickle.dump(pipeline, open("../models/agglomerative_clustering_cmy_k_300_dpi.pkl", 'wb'))